# Phase 2 - Step 5: Feature Engineering

This notebook implements feature engineering, target encoding, feature selection, and the sklearn preprocessing pipeline without target leakage.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import joblib

processed_dir = Path("data/processed")
docs_dir = Path("docs")
docs_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(processed_dir / "employee_attrition_processed.csv")
print(f"Loaded processed dataset with shape: {df.shape}")

# 1. Dynamically identify target
target_candidates = [c for c in df.columns if "attrition" in c.lower() or "target" in c.lower()]
target_col = target_candidates[0]
print(f"Target column detected: '{target_col}'")

# Map binary target
y = df[target_col].map({"Yes": 1, "No": 0}).astype(int)
print(f"Target class distribution:\n{y.value_counts(normalize=True) * 100}")

# 2. Feature Engineering (only from verified existing columns)
df_features = df.copy()

# A. Income per year at company
df_features['income_per_year_at_company'] = df_features['MonthlySalary'] * 12.0 / (df_features['YearsAtCompany'] + 1.0)

# B. Promotion gap ratio: years since last promotion relative to tenure
df_features['promotion_gap_ratio'] = (2026.0 - df_features['LastPromotionYear']) / (df_features['YearsAtCompany'] + 1.0)

# C. Overtime ratio (assuming 160 standard working hours/month)
df_features['overtime_ratio'] = df_features['OvertimeHoursPerMonth'] / 160.0

# D. Leave utilization (assuming 20 standard annual leaves)
df_features['leave_utilization'] = df_features['LeavesTaken'] / 20.0

# E. Work-life satisfaction index
df_features['work_life_satisfaction'] = df_features['WorkLifeBalanceScore'] * df_features['CustomerSatisfaction']

# Drop non-feature and ID columns to avoid leakage
drop_cols = ['EmployeeID', 'Name', 'PhoneNumber', 'JoiningDate', 'LastLeaveDate', target_col, 'CountryCode']
X = df_features.drop(columns=[c for c in drop_cols if c in df_features.columns])

# Separate numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"Numerical Features ({len(num_cols)}):\n{num_cols}")
print(f"Categorical Features ({len(cat_cols)}):\n{cat_cols}")


Loaded processed dataset with shape: (500, 25)
Target column detected: 'AttritionRisk'
Target class distribution:
AttritionRisk
0    89.0
1    11.0
Name: proportion, dtype: float64
Numerical Features (18):
['Age', 'EducationLevel', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'PerformanceRating', 'CustomerSatisfaction_missing', 'income_per_year_at_company', 'promotion_gap_ratio', 'overtime_ratio', 'leave_utilization', 'work_life_satisfaction']
Categorical Features (5):
['Gender', 'Department', 'JobRole', 'Country', 'LeaveDayName']


In [2]:
# Build reproducible ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

# Test fit transform
X_transformed = preprocessor.fit_transform(X)
print(f"Transformed feature matrix shape: {X_transformed.shape}")

# Generate docs/feature_engineering.md
doc_content = f"""# Enterprise HR AI — Feature Engineering Documentation

This document describes all base and domain-engineered features used for predicting employee attrition risk.

## Target Variable
- **Column**: `{target_col}`
- **Encoding**: `Yes` -> 1 (Attrition Risk), `No` -> 0 (Active/Low Risk)
- **Positive Class Prevalence**: ~11.0% (55 positive cases out of 500 records)

## Dropped Identifier / Leakage Columns
- `EmployeeID`: Unique employee identifier (prevent overfitting)
- `Name`: Free-text name (high cardinality, PII)
- `PhoneNumber`: Contact identifier
- `JoiningDate`: Date string (tenure captured via `YearsAtCompany`)
- `LastLeaveDate`: Date string (leave pattern captured via `LeavesTaken` and `LeaveDayName`)
- `CountryCode`: Redundant with `Country`
- `{target_col}`: Excluded to prevent direct target leakage

## Domain Engineered Features

| Feature Name | Formulation | Business Rationale |
|---|---|---|
| `income_per_year_at_company` | `MonthlySalary * 12 / (YearsAtCompany + 1)` | Measures annual compensation velocity relative to tenure. Stagnant earnings over long tenure elevate departure risk. |
| `promotion_gap_ratio` | `(2026 - LastPromotionYear) / (YearsAtCompany + 1)` | Ratio of years without promotion relative to company tenure. High values flag career stagnation. |
| `overtime_ratio` | `OvertimeHoursPerMonth / 160.0` | Overtime burden relative to standard full-time capacity. High overtime induces burnout. |
| `leave_utilization` | `LeavesTaken / 20.0` | Proportion of annual leave allowance utilized. Unused leaves or extreme leave spikes indicate disengagement or stress. |
| `work_life_satisfaction` | `WorkLifeBalanceScore * CustomerSatisfaction` | Interaction term measuring composite workplace well-being. |

## Preprocessing Pipeline Architecture
- **Numerical Pipeline**: Median Imputation -> Standard Scaling
- **Categorical Pipeline**: Most Frequent Imputation -> One-Hot Encoding (`handle_unknown='ignore'`)
- **Total Input Features**: {len(num_cols) + len(cat_cols)} (Transformed matrix width: {X_transformed.shape[1]})
"""

with open(docs_dir / "feature_engineering.md", "w", encoding="utf-8") as f:
    f.write(doc_content)

print(f"Generated {docs_dir / 'feature_engineering.md'}")


Transformed feature matrix shape: (500, 53)
Generated docs\feature_engineering.md
